# Project Eco: train the YOLO26 window detector on a free GPU

**Before you start:** menu **Runtime → Change runtime type → T4 GPU → Save**.
Then **Runtime → Run all**. When asked, paste your **Roboflow Private API key** and press Enter.

It takes about 30–40 minutes. At the end, your browser downloads **`yolo26_windows.pt`**.
Put that file in `Green-House/eco-engine/models/` on your PC (or tell Claude where it is).

Dataset: [WindowDitection](https://universe.roboflow.com/gautam-parmar-2u4ak/windowditection), CC BY 4.0.

In [ ]:
# 1. Check that a GPU is attached (otherwise training would take many hours)
import torch
assert torch.cuda.is_available(), "No GPU: use Runtime > Change runtime type > T4 GPU, then Run all again."
print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
# 2. Install the same Ultralytics version the Project Eco app uses
!pip -q install ultralytics==8.4.126

In [ ]:
# 3. Download the windows dataset from Roboflow (your key is used only here and is not saved)
import io, json, urllib.request, zipfile
from getpass import getpass
from pathlib import Path

key = getpass("Paste your Roboflow Private API key and press Enter: ").strip()
with urllib.request.urlopen(f"https://api.roboflow.com/gautam-parmar-2u4ak/windowditection/1/yolov8?api_key={key}", timeout=120) as r:
    link = json.load(r)["export"]["link"]
del key
data = urllib.request.urlopen(link, timeout=600).read()
target = Path("/content/datasets/windows")
target.mkdir(parents=True, exist_ok=True)
zipfile.ZipFile(io.BytesIO(data)).extractall(target)
for split in ("train", "valid", "test"):
    print(split, len(list((target / split / "images").glob("*"))), "images")

In [ ]:
# 4. Train YOLO26s (one class: window). About 30 minutes on a T4 GPU.
from ultralytics import YOLO

model = YOLO("yolo26s.pt")
model.train(
    data="/content/datasets/windows/data.yaml", epochs=80, imgsz=640, batch=16, device=0,
    single_cls=True, patience=20, cos_lr=True, workers=2,
    project="/content/runs", name="windows-yolo26s", exist_ok=True, plots=False,
)

In [ ]:
# 5. Score it on the held-out test images (never seen during training)
best = YOLO("/content/runs/windows-yolo26s/weights/best.pt")
m = best.val(data="/content/datasets/windows/data.yaml", split="test", single_cls=True, imgsz=640, plots=False)
print(f"TEST  mAP50={m.box.map50:.3f}  mAP50-95={m.box.map:.3f}  precision={m.box.mp:.3f}  recall={m.box.mr:.3f}")

In [ ]:
# 6. Download the trained model to your PC
import shutil
from google.colab import files

shutil.copy("/content/runs/windows-yolo26s/weights/best.pt", "/content/yolo26_windows.pt")
files.download("/content/yolo26_windows.pt")